# Tennis Court Keypoint Detection - 3-Stage Training

**Architecture:** MobileNetV3-Small + Heatmap Decoder

**Strategy:**
- Stage 1: Pretrain on broadcast data (court geometry)
- Stage 2: Mixed training (broadcast + phone, 50:50 oversampling)
- Stage 3: Fine-tune on phone data only (low LR)

**Instructions:**
1. Runtime > Change runtime type > T4 GPU
2. Run all cells in order
3. Results will be saved to Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Check GPU
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Install dependencies
!pip install -q albumentations

In [ ]:
# Clone repo and setup
!git clone https://github.com/kokoro456/SHOT-AI.git /content/shot-ai 2>/dev/null || (cd /content/shot-ai && git pull)
import sys
sys.path.insert(0, '/content/shot-ai/ml/src')
print('Repo ready')

In [ ]:
# Download and extract data from Google Drive
import os, zipfile, json

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

# Phone data (labeled by user)
PHONE_ZIP = '/content/drive/MyDrive/SHOT-AI/youtube_labeled_data.zip'
if os.path.exists(PHONE_ZIP):
    with zipfile.ZipFile(PHONE_ZIP, 'r') as z:
        z.extractall(f'{DATA_DIR}/phone')
    print(f'Phone data extracted')
else:
    print(f'ERROR: {PHONE_ZIP} not found! Upload youtube_labeled_data.zip to Drive/SHOT-AI/')

# Broadcast data (download from source)
BROADCAST_DIR = f'{DATA_DIR}/broadcast'
os.makedirs(BROADCAST_DIR, exist_ok=True)

BROADCAST_ZIP = f'{BROADCAST_DIR}/dataset.zip'
if not os.path.exists(BROADCAST_ZIP):
    print('Downloading broadcast dataset (7.3GB)...')
    !pip install -q gdown
    import gdown
    gdown.download(id='1lhAaeQCmk2y440PmagA0KmIVBIysVMwu', output=BROADCAST_ZIP, quiet=False)

if os.path.exists(BROADCAST_ZIP) and not os.path.exists(f'{BROADCAST_DIR}/data/images'):
    print('Extracting broadcast dataset...')
    with zipfile.ZipFile(BROADCAST_ZIP, 'r') as z:
        z.extractall(BROADCAST_DIR)
    print('Broadcast data extracted')

# Count files
phone_frames = len([f for f in os.listdir(f'{DATA_DIR}/phone/frames') if f.endswith('.jpg')])
broadcast_imgs = len([f for f in os.listdir(f'{BROADCAST_DIR}/data/images') if f.endswith('.png')])
print(f'\nPhone frames: {phone_frames}')
print(f'Broadcast images: {broadcast_imgs}')

In [ ]:
# Convert broadcast annotations to SHOT format
import json

broadcast_ann_path = f'{DATA_DIR}/broadcast/annotations_broadcast.json'

if not os.path.exists(broadcast_ann_path):
    train_data = json.load(open(f'{BROADCAST_DIR}/data/data_train.json'))
    val_data = json.load(open(f'{BROADCAST_DIR}/data/data_val.json'))

    # Mapping from yastrebksv indices to SHOT keypoint IDs
    # yastrebksv has 14 keypoints, we need points 9-16
    YASTREBKSV_TO_SHOT = {10:'9', 13:'10', 11:'11', 2:'12', 5:'13', 7:'15', 3:'16'}

    all_annotations = []
    for entry in train_data + val_data:
        kps_raw = entry['kps']
        keypoints = {}
        for yidx, shot_id in YASTREBKSV_TO_SHOT.items():
            if yidx < len(kps_raw):
                keypoints[shot_id] = {
                    'x': round(kps_raw[yidx][0]/1280, 6),
                    'y': round(kps_raw[yidx][1]/720, 6),
                    'visible': True
                }
            else:
                keypoints[shot_id] = {'x': 0, 'y': 0, 'visible': False}
        # Pt14 = midpoint of baseline
        if 2 < len(kps_raw) and 3 < len(kps_raw):
            keypoints['14'] = {
                'x': round((kps_raw[2][0]+kps_raw[3][0])/2/1280, 6),
                'y': round((kps_raw[2][1]+kps_raw[3][1])/2/720, 6),
                'visible': True
            }
        all_annotations.append({'image': entry['id']+'.png', 'keypoints': keypoints})

    json.dump(all_annotations, open(broadcast_ann_path, 'w'), indent=2)
    print(f'Converted {len(all_annotations)} broadcast annotations')
else:
    data = json.load(open(broadcast_ann_path))
    print(f'Broadcast annotations already exist: {len(data)}')

In [ ]:
# ============================================================
# 3-STAGE TRAINING
# ============================================================
import sys, time, json, os
sys.path.insert(0, '/content/shot-ai/ml/src')

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, random_split, ConcatDataset, Subset, WeightedRandomSampler

from dataset import CourtKeypointDataset
from model_heatmap import HeatmapKeypointModel, create_heatmap_model, NUM_KEYPOINTS, HEATMAP_SIZE
from augmentations_v2 import get_strong_augmentation, get_phone_augmentation, get_val_augmentation

KP_NAMES = ['Pt9(SL)', 'Pt10(SC)', 'Pt11(SR)', 'Pt12(DL)',
            'Pt13(BL)', 'Pt14(BC)', 'Pt15(BR)', 'Pt16(DR)']

device = torch.device('cuda')
print(f'Device: {device} ({torch.cuda.get_device_name(0)})')

# === Load datasets ===
phone_full = CourtKeypointDataset(
    annotation_file=f'{DATA_DIR}/phone/labeled_annotations.json',
    image_dir=f'{DATA_DIR}/phone/frames',
    input_size=256,
)
print(f'Phone: {len(phone_full)} images')

# Split: 80% train, 20% test
test_size = int(len(phone_full) * 0.2)
train_size = len(phone_full) - test_size
gen = torch.Generator().manual_seed(42)
phone_train, phone_test = random_split(phone_full, [train_size, test_size], generator=gen)
print(f'Phone train: {len(phone_train)}, test: {len(phone_test)}')

broadcast_full = CourtKeypointDataset(
    annotation_file=broadcast_ann_path,
    image_dir=f'{BROADCAST_DIR}/data/images',
    input_size=256,
)
print(f'Broadcast: {len(broadcast_full)} images')

# Val loader
phone_test.dataset.augmentation = get_val_augmentation(256)
val_loader = DataLoader(phone_test, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
# Training functions
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    for batch in loader:
        images = batch['image'].to(device)
        gt_kps = batch['keypoints'].to(device)
        gt_vis = batch['visibility'].to(device)

        optimizer.zero_grad()
        pred_hm = model(images)
        target_hm = HeatmapKeypointModel.generate_heatmap_targets(gt_kps, gt_vis, HEATMAP_SIZE, sigma=3.0)
        vis_mask = gt_vis.unsqueeze(2).unsqueeze(3)
        loss = ((pred_hm - target_hm) ** 2 * vis_mask).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    per_kp_errors = torch.zeros(NUM_KEYPOINTS)
    per_kp_counts = torch.zeros(NUM_KEYPOINTS)
    total_loss = 0
    for batch in loader:
        images = batch['image'].to(device)
        gt_kps = batch['keypoints'].to(device)
        gt_vis = batch['visibility'].to(device)

        pred_hm = model(images)
        target_hm = HeatmapKeypointModel.generate_heatmap_targets(gt_kps, gt_vis, HEATMAP_SIZE, sigma=3.0)
        vis_mask = gt_vis.unsqueeze(2).unsqueeze(3)
        loss = ((pred_hm - target_hm) ** 2 * vis_mask).mean()
        total_loss += loss.item()

        pred_coords = HeatmapKeypointModel.heatmaps_to_coords(pred_hm)
        pixel_error = torch.sqrt(((pred_coords - gt_kps) ** 2).sum(dim=2)) * 256
        for i in range(NUM_KEYPOINTS):
            mask = gt_vis[:, i] > 0
            if mask.any():
                per_kp_errors[i] += pixel_error[mask, i].sum().item()
                per_kp_counts[i] += mask.sum().item()

    per_kp_avg = per_kp_errors / (per_kp_counts + 1e-8)
    return total_loss / len(loader), per_kp_avg, per_kp_avg.mean().item()

def run_stage(name, model, train_loader, val_loader, device,
              epochs, lr, patience, save_dir):
    print(f'\n{"="*60}')
    print(f'  {name}')
    print(f'  LR={lr}, Epochs={epochs}, Patience={patience}')
    print(f'{"="*60}')

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    os.makedirs(save_dir, exist_ok=True)

    best_error = float('inf')
    best_per_kp = None
    patience_cnt = 0

    for epoch in range(epochs):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_loss, per_kp, mean_err = evaluate(model, val_loader, device)
        scheduler.step()
        elapsed = time.time() - t0

        if (epoch+1) % 5 == 0 or epoch == 0:
            print(f'  Epoch {epoch+1:3d} | Train: {train_loss:.6f} | Val: {val_loss:.6f} | Mean: {mean_err:.2f}px | {elapsed:.1f}s')

        if mean_err < best_error:
            best_error = mean_err
            best_per_kp = per_kp.clone()
            patience_cnt = 0
            torch.save(model.state_dict(), f'{save_dir}/best_model.pth')
        else:
            patience_cnt += 1

        if patience_cnt >= patience:
            print(f'  Early stopping at epoch {epoch+1}')
            break

    print(f'  Best: {best_error:.2f}px')
    for i, n in enumerate(KP_NAMES):
        print(f'    {n}: {best_per_kp[i]:.2f}px')

    model.load_state_dict(torch.load(f'{save_dir}/best_model.pth', weights_only=True))
    return best_error, best_per_kp

print('Training functions ready')

In [ ]:
# ============================================================
# STAGE 1: Pretrain on broadcast data
# ============================================================
SAVE_BASE = '/content/drive/MyDrive/SHOT-AI/models/3stage'

model = create_heatmap_model(pretrained=True).to(device)
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

broadcast_full.augmentation = get_strong_augmentation(256)
broadcast_loader = DataLoader(broadcast_full, batch_size=32, shuffle=True, num_workers=2)

s1_error, s1_per_kp = run_stage(
    'STAGE 1: Pretrain on Broadcast (8.8K)',
    model, broadcast_loader, val_loader, device,
    epochs=50, lr=1e-3, patience=15,
    save_dir=f'{SAVE_BASE}/stage1'
)

In [ ]:
# ============================================================
# STAGE 2: Mixed training (broadcast + phone, target oversampled)
# ============================================================
broadcast_full.augmentation = get_strong_augmentation(256)
phone_train.dataset.augmentation = get_phone_augmentation(256)

combined = ConcatDataset([broadcast_full, phone_train])
n_bc = len(broadcast_full)
n_ph = len(phone_train)
target_ratio = 0.5  # 50:50 sampling

w_bc = (1 - target_ratio) / n_bc
w_ph = target_ratio / n_ph
weights = [w_bc] * n_bc + [w_ph] * n_ph
sampler = WeightedRandomSampler(weights, num_samples=n_bc + n_ph, replacement=True)
mixed_loader = DataLoader(combined, batch_size=32, sampler=sampler, num_workers=2)

print(f'Mixed: {n_bc} broadcast + {n_ph} phone (50:50 oversampling)')

s2_error, s2_per_kp = run_stage(
    'STAGE 2: Mixed Training (Target Oversampled)',
    model, mixed_loader, val_loader, device,
    epochs=80, lr=5e-4, patience=20,
    save_dir=f'{SAVE_BASE}/stage2'
)

In [ ]:
# ============================================================
# STAGE 3: Fine-tune on phone data only (low LR)
# ============================================================
phone_train.dataset.augmentation = get_phone_augmentation(256)
phone_loader = DataLoader(phone_train, batch_size=32, shuffle=True, num_workers=2)

s3_error, s3_per_kp = run_stage(
    'STAGE 3: Fine-tune on Phone Only',
    model, phone_loader, val_loader, device,
    epochs=40, lr=1e-4, patience=15,
    save_dir=f'{SAVE_BASE}/stage3_final'
)

In [ ]:
# ============================================================
# FINAL REPORT
# ============================================================
print(f'\n{"="*60}')
print(f'  3-STAGE TRAINING COMPLETE')
print(f'{"="*60}')
print(f'\n{"Stage":<40} {"Mean Error":>10}')
print('-' * 55)
print(f'  Stage 1 (Broadcast pretrain)           {s1_error:>8.2f}px')
print(f'  Stage 2 (Mixed + oversampling)         {s2_error:>8.2f}px')
print(f'  Stage 3 (Phone fine-tune) [FINAL]      {s3_error:>8.2f}px')

print(f'\nFinal per-keypoint errors:')
for i, n in enumerate(KP_NAMES):
    print(f'  {n}: {s3_per_kp[i]:.2f}px')

# Save report
report = {
    'stage1_error': s1_error,
    'stage2_error': s2_error,
    'stage3_error': s3_error,
    'final_per_kp': {KP_NAMES[i]: round(s3_per_kp[i].item(), 2) for i in range(NUM_KEYPOINTS)},
}
json.dump(report, open(f'{SAVE_BASE}/training_report.json', 'w'), indent=2)
print(f'\nReport saved to {SAVE_BASE}/training_report.json')
print(f'Final model saved to {SAVE_BASE}/stage3_final/best_model.pth')

In [ ]:
# ============================================================
# VISUAL TEST: Show predictions on test images
# ============================================================
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

model.eval()
COLORS = ['#FF4444', '#FF8800', '#FFCC00', '#44FF44',
          '#00CCFF', '#4488FF', '#8844FF', '#FF44FF']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

phone_test.dataset.augmentation = get_val_augmentation(256)

MEAN = np.array([0.485, 0.456, 0.406])
STD = np.array([0.229, 0.224, 0.225])

for idx in range(min(8, len(phone_test))):
    sample = phone_test[idx]
    img = sample['image'].unsqueeze(0).to(device)
    gt_kps = sample['keypoints']
    gt_vis = sample['visibility']

    with torch.no_grad():
        pred_hm = model(img)
        pred_coords = HeatmapKeypointModel.heatmaps_to_coords(pred_hm)[0].cpu()

    # Denormalize image
    img_np = sample['image'].numpy().transpose(1, 2, 0)
    img_np = (img_np * STD + MEAN).clip(0, 1)

    axes[idx].imshow(img_np)
    for k in range(NUM_KEYPOINTS):
        if gt_vis[k] > 0:
            gx, gy = gt_kps[k, 0].item() * 256, gt_kps[k, 1].item() * 256
            px, py = pred_coords[k, 0].item() * 256, pred_coords[k, 1].item() * 256
            axes[idx].plot(gx, gy, 'o', color=COLORS[k], markersize=8, label=f'GT {KP_NAMES[k]}')
            axes[idx].plot(px, py, 'x', color=COLORS[k], markersize=10, markeredgewidth=2)
            axes[idx].plot([gx, px], [gy, py], '-', color=COLORS[k], alpha=0.5)
    axes[idx].set_title(f'Test #{idx}')
    axes[idx].axis('off')

plt.suptitle(f'Predictions (o=GT, x=Pred) | Mean Error: {s3_error:.2f}px', fontsize=14)
plt.tight_layout()
plt.savefig(f'{SAVE_BASE}/visual_test.png', dpi=150)
plt.show()
print(f'Visual test saved to {SAVE_BASE}/visual_test.png')